# Bayesian Networks and Autoregressive Language Models
This notebook is the solution for the lab `BN_lab.pdf`: first- and second-order word-level language models, viewed as Bayesian networks.

**LLM use:** the `NGramLM` class was generated with Claude (Anthropic) from the prompts in Parts V and XII, then inspected, corrected and tested. The corrections are described in Part VI and the final reflection. No ML library or pretrained model is used – only Python data structures and `random`.

## Part I: From probability to language

**Question 1 – Why is the chain-rule decomposition useful for generating text?**

It turns one huge joint distribution over whole sentences into a product of *next-word* distributions, each conditioned only on what has already been produced:
$P(X_1,\dots,X_T) = P(X_1)\prod_{t\ge2} P(X_t \mid X_1,\dots,X_{t-1})$.
That gives a direct generation procedure: sample $X_1$, then sample $X_2$ given $X_1$, and so on, one token at a time, left to right. We never need to enumerate or normalise over all possible sentences; at each step we only need one distribution over the vocabulary. The decomposition is exact (no assumption yet), and the same factors also let us score any sentence by multiplying the conditional probabilities along it.

## Part II: A Bayesian network for text

**Question 2 – What independence assumption does the chain $X_1 \to X_2 \to \dots \to X_T$ make?**

Each word is conditionally independent of all earlier words given the immediately preceding word (the first-order Markov assumption):

$$P(X_t \mid X_1, \dots, X_{t-1}) = P(X_t \mid X_{t-1}), \qquad \text{i.e.}\quad X_t \perp\!\!\!\perp \{X_1,\dots,X_{t-2}\} \mid X_{t-1}.$$

In BN terms: every node is independent of its non-descendants given its parent, and each node has exactly one parent, $X_{t-1}$.

## Part III: Build the dataset

In [1]:
import random
from collections import Counter, defaultdict

RAW = """
the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug
"""

START, END = "<START>", "<END>"

def tokenise(text):
    return [s.lower().split() for s in text.strip().splitlines() if s.strip()]

sentences = tokenise(RAW)
for s in sentences:
    print(" ".join([START] + s + [END]))
vocab = sorted({w for s in sentences for w in s})
print("\nvocabulary:", vocab, f"({len(vocab)} words)")

<START> the cat sat on the mat <END>
<START> the cat sat on the rug <END>
<START> the dog sat on the mat <END>
<START> the dog ran to the park <END>
<START> the cat ran to the park <END>
<START> the dog sat on the rug <END>

vocabulary: ['cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', 'to'] (10 words)


## Part IV: The conditional probability table (by hand)

**Question 3** – counting each "current → next" pair across the six sentences (with `<START>`/`<END>`):

| current word $w$ | $P(\cdot \mid w)$ |
|---|---|
| `<START>` | the: 6/6 = 1 |
| the | cat: 3/12, dog: 3/12, mat: 2/12, rug: 2/12, park: 2/12 |
| cat | sat: 2/3, ran: 1/3 |
| dog | sat: 2/3, ran: 1/3 |
| sat | on: 4/4 = 1 |
| ran | to: 2/2 = 1 |
| on | the: 1 |
| to | the: 1 |
| mat, rug, park | `<END>`: 1 |

"the" occurs twice per sentence, so it is the context 12 times.

**Zero-probability transitions** (anything not in the table has probability 0), for example: $P(\text{sat} \mid \text{the}) = 0$, $P(\text{the} \mid \text{cat}) = 0$, $P(\text{on} \mid \text{cat}) = 0$, $P(\text{to} \mid \text{sat}) = 0$, $P(\text{on} \mid \text{ran}) = 0$, $P(\langle END\rangle \mid \text{cat}) = 0$. So "the cat ran on the mat" gets probability 0 even though it is a perfectly good English sentence. Also `<END>` has no outgoing transitions at all, so its distribution is undefined.

The code version is checked against this hand table in Part VII.

## Part V: Ask an LLM to implement the model

**Prompt used:**

> Write a simple Python implementation of a first-order autoregressive language model. The model should:
> 1. take a list of tokenised sentences as training data;
> 2. count transitions between consecutive tokens;
> 3. construct the conditional distribution $P(X_t \mid X_{t-1})$;
> 4. display the probabilities for a specified previous token;
> 5. predict the most probable next token;
> 6. generate a sentence by repeatedly sampling the next token;
> 7. stop when the `<END>` token is generated.
>
> Do not use a machine-learning library or a pretrained language model. Use ordinary Python data structures and random sampling.
> Write it as a class with an `order` parameter (the number of previous tokens used as context), so that I can reuse it for a second-order model later. Support both greedy and sampling generation.

The code below is the generated class **after** my corrections (marked `# FIX`, explained in Part VI).

In [2]:
class NGramLM:
    """
    Autoregressive n-gram language model.
      order = 1 : P(X_t | X_{t-1})            (bigram,  BN chain X_{t-1} -> X_t)
      order = 2 : P(X_t | X_{t-2}, X_{t-1})   (trigram, BN X_{t-2} -> X_t <- X_{t-1})
    Every sentence is padded with `order` copies of <START>, so the first word is
    modelled as P(X_1 | <START>) (and P(X_2 | <START>, X_1) for order 2).
    """

    def __init__(self, order=1):
        self.order = order
        self.counts = defaultdict(Counter)    # context tuple -> Counter of next tokens
        self.probs = {}                       # context tuple -> {next token: probability}

    def fit(self, sentences):
        for s in sentences:
            tokens = [START] * self.order + s + [END]
            for i in range(self.order, len(tokens)):
                context = tuple(tokens[i - self.order:i])
                self.counts[context][tokens[i]] += 1
        # P(w | context) = C(context, w) / sum_k C(context, k)
        self.probs = {
            ctx: {w: c / sum(nxt.values()) for w, c in nxt.items()}
            for ctx, nxt in self.counts.items()
        }
        return self

    def distribution(self, context):
        context = tuple(context)[-self.order:]
        if context not in self.probs:         # FIX 1: unseen context -> explicit, not KeyError
            return None
        return self.probs[context]

    def show(self, context):
        dist = self.distribution(context)
        label = ", ".join(context[-self.order:]) if not isinstance(context, str) else context
        if dist is None:
            print(f"P(. | {label}): context never observed")
            return
        print(f"P(. | {label}):")
        for w, p in sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])):
            print(f"   {w:<8} {p:.4f}")

    def predict(self, context):
        dist = self.distribution(context)
        if dist is None:
            return None
        best = max(dist.values())
        return sorted(w for w, p in dist.items() if p == best)   # FIX 2: return ALL tied argmaxes

    def next_token(self, context, mode, rng):
        dist = self.distribution(context)
        if dist is None:
            return END                        # FIX 1: unseen context -> stop the sentence
        if mode == "greedy":
            return self.predict(context)[0]   # deterministic tie-break: alphabetical
        words = list(dist)
        return rng.choices(words, weights=[dist[w] for w in words], k=1)[0]

    def generate(self, mode="sample", rng=None, max_len=20):
        rng = rng or random.Random()
        context = [START] * self.order
        out = []
        while len(out) < max_len:             # FIX 3: length cap (greedy can loop forever)
            w = self.next_token(context, mode, rng)
            if w == END:
                return " ".join(out)
            out.append(w)
            context = (context + [w])[-self.order:]
        return " ".join(out) + " ...[max length reached]"

    def sentence_prob(self, sentence):
        """Chain rule: product of conditional probabilities along the sentence (incl. <END>)."""
        tokens = [START] * self.order + sentence.split() + [END]
        p = 1.0
        for i in range(self.order, len(tokens)):
            dist = self.distribution(tokens[i - self.order:i]) or {}
            p *= dist.get(tokens[i], 0.0)
        return p


bigram = NGramLM(order=1).fit(sentences)

## Part VI: Inspect the LLM-generated code

**Question 4 – Where are the transition counts stored?** In `self.counts`, a `defaultdict(Counter)` mapping each context (a tuple of the previous `order` tokens, e.g. `("the",)`) to a `Counter` of the tokens observed after it. They are filled in `fit`.

**Question 5 – Where is $P(X_t \mid X_{t-1})$ computed?** In `fit`, in the dictionary comprehension that builds `self.probs`: each count is divided by the total count for that context, $C(w_i, w_j)/\sum_k C(w_i, w_k)$. `distribution()` just looks it up.

**Question 6 – How is the next word chosen?** `next_token` supports both. In `"greedy"` mode it always takes the argmax of the distribution (via `predict`). In `"sample"` mode it uses `rng.choices` with the probabilities as weights, i.e. it draws $w \sim P(\cdot \mid w_{prev})$. Greedy is deterministic: the same context always gives the same word, so only one sentence can ever come out. Sampling picks less likely words in proportion to their probability, so repeated runs give different sentences whose frequencies follow the model.

**Question 7 – What happens for an unseen context?** A direct lookup `self.probs[context]` raises a `KeyError` (e.g. the distribution after `<END>`, or after any out-of-vocabulary word). With FIX 1, `distribution()` returns `None`, `show()` reports that the context was never observed, and generation ends the sentence. This is a modelling decision, not just error handling: a pure count-based model has *no* distribution for an unseen context. Smoothing (e.g. add-one) or backing off to a lower-order model would be the principled alternatives.

**Issues found on inspection and fixed (marked `# FIX` in the code):**
1. **Unseen context** – a plain dictionary lookup crashes with `KeyError`; now explicit `None` / end of sentence, as above.
2. **Ties in the argmax** – the usual one-liner `max(dist, key=dist.get)` returns whichever tied word comes first in the dictionary ("cat", only because it was seen first). For "the" the true argmax is the *set* {cat, dog}, so `predict` returns all tied words and greedy generation uses an explicit, documented tie-break (alphabetical).
3. **No length limit** – a `while next != <END>` loop never terminates under greedy decoding on this data (shown in Part X). Added `max_len`.

## Part VII: Test the probability model

In [3]:
def normalisation_test(model, tol=1e-12):
    ok = True
    for ctx, dist in sorted(model.probs.items()):
        total = sum(dist.values())
        flag = "OK" if abs(total - 1) < tol else "FAIL"
        ok &= flag == "OK"
        print(f"{' '.join(ctx):<18} sum = {total:.12f}  {flag}")
    return ok

print("First-order model:")
assert normalisation_test(bigram)

# The code's CPT must equal the hand-computed table from Question 3
from fractions import Fraction as Fr
hand = {
    "the": {"cat": Fr(3, 12), "dog": Fr(3, 12), "mat": Fr(2, 12), "rug": Fr(2, 12), "park": Fr(2, 12)},
    "cat": {"sat": Fr(2, 3), "ran": Fr(1, 3)},
    "dog": {"sat": Fr(2, 3), "ran": Fr(1, 3)},
    "sat": {"on": Fr(1)},
    "ran": {"to": Fr(1)},
}
for w, expected in hand.items():
    got = bigram.distribution([w])
    assert set(got) == set(expected) and all(abs(got[k] - float(v)) < 1e-12 for k, v in expected.items()), w
print("\nCode CPT matches the hand-computed CPT for: the, cat, dog, sat, ran")

First-order model:
<START>            sum = 1.000000000000  OK
cat                sum = 1.000000000000  OK
dog                sum = 1.000000000000  OK
mat                sum = 1.000000000000  OK
on                 sum = 1.000000000000  OK
park               sum = 1.000000000000  OK
ran                sum = 1.000000000000  OK
rug                sum = 1.000000000000  OK
sat                sum = 1.000000000000  OK
the                sum = 1.000000000000  OK
to                 sum = 1.000000000000  OK

Code CPT matches the hand-computed CPT for: the, cat, dog, sat, ran


In [4]:
# Demonstrating what Question 8 is about: a plausible bug - add-one (Laplace) smoothing
# implemented only half-way. The denominator adds |V|, but the +1 is only given to next words
# that were actually observed, so the mass reserved for unseen words is never in the table.
def buggy_smoothed(model, vocab_size):
    return {ctx: {w: (c + 1) / (sum(nxt.values()) + vocab_size) for w, c in nxt.items()}
            for ctx, nxt in model.counts.items()}

for ctx, d in sorted(buggy_smoothed(bigram, len(vocab) + 1).items()):
    print(f"{' '.join(ctx):<8} sum = {sum(d.values()):.4f}")

<START>  sum = 0.4118
cat      sum = 0.3571
dog      sum = 0.3571
mat      sum = 0.2308
on       sum = 0.3333
park     sum = 0.2308
ran      sum = 0.2308
rug      sum = 0.2308
sat      sum = 0.3333
the      sum = 0.7391
to       sum = 0.2308


**Question 8 – A total of 0.87?** It means the distribution for that context is not normalised, so the implementation is wrong: some probability mass is missing. Typical causes: the wrong denominator (e.g. dividing by how often the word occurs anywhere, or adding a smoothing term to the denominator only), transitions missing from the numerator (an off-by-one that drops a `→ <END>` step), or probability mass assigned to words that were never written into the table. The half-implemented smoothing above is exactly this kind of bug: every context sums to well below 1 (e.g. 0.74 for "the", 0.36 for "cat"), because the probability reserved for unseen next words was never actually added to the table. Sampling code that normalises its weights (like `random.choices`) would hide the problem during generation, but every sentence probability computed with the chain rule would be wrong. Our model passes for every context.

## Part VIII: Predicting the next word

In [5]:
for w in ["the", "cat", "dog", "sat", "ran", "on", "to", START]:
    bigram.show([w])
    print(f"   argmax: {bigram.predict([w])}\n")

P(. | the):
   cat      0.2500
   dog      0.2500
   mat      0.1667
   park     0.1667
   rug      0.1667
   argmax: ['cat', 'dog']

P(. | cat):
   sat      0.6667
   ran      0.3333
   argmax: ['sat']

P(. | dog):
   sat      0.6667
   ran      0.3333
   argmax: ['sat']

P(. | sat):
   on       1.0000
   argmax: ['on']

P(. | ran):
   to       1.0000
   argmax: ['to']

P(. | on):
   the      1.0000
   argmax: ['the']

P(. | to):
   the      1.0000
   argmax: ['the']

P(. | <START>):
   the      1.0000
   argmax: ['the']



| context $w$ | $\arg\max_v P(v \mid w)$ | probability |
|---|---|---|
| the | **cat and dog (tie)** | 0.25 each |
| cat | sat | 0.667 |
| dog | sat | 0.667 |
| sat | on | 1.0 |
| ran | to | 1.0 |
| on, to | the | 1.0 |
| `<START>` | the | 1.0 |

**Question 9 – Do the predictions match what I'd expect?** Partly. After "sat" → "on" and "ran" → "to" match intuition. But the model is certain that after "on" comes "the" (a human would allow "a", "my", …), it treats "the" as equally likely to be followed by "cat" or "dog" but never by "big" or "old", and it has no idea that "the" at the start of a sentence (a subject) and "the" after "on" (a location) should be followed by different words – after "on the" it still predicts "cat" or "dog". A probability model only reflects the frequencies in its training data under its independence assumption; human expectations come from far more data, meaning and longer context. High probability under the model means "frequent in this tiny corpus given one previous word", not "what a person would say".

## Part IX: Generate text (20 sampled sentences)

In [6]:
rng = random.Random(42)
generated = [bigram.generate("sample", rng) for _ in range(20)]
train_set = {" ".join(s) for s in sentences}
for i, s in enumerate(generated, 1):
    tag = "" if s in train_set else "   <- not in training data"
    print(f"{i:>2}. {s}{tag}")

with open("generated_sentences.txt", "w") as f:
    f.write("\n".join(generated) + "\n")
print("\nsaved to generated_sentences.txt")
print(f"novel sentences: {sum(s not in train_set for s in generated)}/20, distinct: {len(set(generated))}/20")

 1. the cat sat on the dog ran to the cat sat on the cat sat on the dog ran to ...[max length reached]   <- not in training data
 2. the dog sat on the mat
 3. the park   <- not in training data
 4. the dog sat on the rug
 5. the park   <- not in training data
 6. the cat sat on the cat sat on the mat   <- not in training data
 7. the mat   <- not in training data
 8. the dog sat on the mat
 9. the rug   <- not in training data
10. the dog sat on the mat
11. the park   <- not in training data
12. the mat   <- not in training data
13. the mat   <- not in training data
14. the mat   <- not in training data
15. the mat   <- not in training data
16. the rug   <- not in training data
17. the cat sat on the cat sat on the park   <- not in training data
18. the dog ran to the dog sat on the park   <- not in training data
19. the rug   <- not in training data
20. the park   <- not in training data

saved to generated_sentences.txt
novel sentences: 16/20, distinct: 9/20


16 of the 20 sentences never appear in the training data. Two failure patterns show up:
- **Too short:** "the mat", "the park", "the rug" (11 of 20). After `<START> the`, the model gives mat/rug/park a combined probability of 0.5, because it cannot tell sentence-initial "the" from "on the"/"to the" – so half of all sentences end after two words.
- **Run-ons:** "the cat sat on the cat sat on the mat", "the dog ran to the dog sat on the park". After "on the", it can continue with "cat"/"dog" because those followed "the" at sentence start. Every adjacent pair is locally plausible, but the sentence is not; sentence 1 even hit the length cap.

## Part X: Greedy vs sampling

In [7]:
print("Mode A - greedy:")
for i in range(5):
    print(f"  {i+1}. {bigram.generate('greedy')}")

print("\nMode B - sampling:")
rng = random.Random(7)
for i in range(5):
    print(f"  {i+1}. {bigram.generate('sample', rng)}")

Mode A - greedy:
  1. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on ...[max length reached]
  2. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on ...[max length reached]
  3. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on ...[max length reached]
  4. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on ...[max length reached]
  5. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on ...[max length reached]

Mode B - sampling:
  1. the cat sat on the mat
  2. the cat sat on the rug
  3. the cat sat on the mat
  4. the park
  5. the cat sat on the rug


**Question 10 – Which mode varies more, and why?** Sampling. Greedy is a deterministic function of the context, so all five greedy outputs are identical. Worse, here greedy **never terminates**: `the → cat (tie-break) → sat → on → the → cat → …` is a cycle, because the most likely word after "the" is never an ending word (mat/rug/park each have only 2/12). It only stops at the length cap – this is the bug that FIX 3 addresses. Sampling draws from the whole distribution, so after "the" it picks mat/rug/park 50% of the time and ends the sentence, and it varies between cat/dog, sat/ran and the three locations, producing different sentences each time.

## Part XI: A second-order Bayesian network

**Question 11 – How does the second-order model differ?**
1. **Graph structure:** each $X_t$ now has two parents, $X_{t-2} \to X_t \leftarrow X_{t-1}$ (plus the chain edge $X_{t-1} \to X_t$ as before), instead of one.
2. **CPT:** indexed by *pairs* of previous words. With $V$ possible tokens the table has up to $V^2$ rows (contexts) of $V$ entries each, instead of $V$ rows.
3. **Context:** two words instead of one, so it can tell "the" at sentence start (`<START> the`) from "the" after a preposition (`on the`, `to the`).
4. **Data needed:** much more. Each pair-context has to be observed for its distribution to be estimated, and each one is seen far less often than a single word, so more contexts are unseen or estimated from 1–2 counts.

## Part XII: Use the LLM again

**What should change in the probabilistic model (stated before asking):** the context goes from $X_{t-1}$ to $(X_{t-2}, X_{t-1})$; counts become triple counts $C(w_i, w_j, w_k)$; $P(w_k \mid w_i, w_j) = C(w_i,w_j,w_k) / \sum_m C(w_i,w_j,w_m)$; sentences need two `<START>` pads so $X_1$ and $X_2$ have a full context; generation keeps the last two tokens. Normalisation must still hold for every context.

**Prompt used:**
> Modify the existing first-order autoregressive model into a second-order model. The model should estimate $P(X_t \mid X_{t-2}, X_{t-1})$. Represent the model using counts of observed triples and use these counts to construct conditional probability distributions. Do not replace the model with a neural network or a pretrained language model.

Because the class was already parameterised by `order`, the LLM's change amounted to padding with `order` start tokens and using tuple contexts – i.e. `NGramLM(order=2)`. Inspected and tested below.

In [8]:
trigram = NGramLM(order=2).fit(sentences)

print("Second-order model:")
assert normalisation_test(trigram)

print()
for ctx in [(START, "the"), ("on", "the"), ("to", "the"), ("the", "cat"), ("cat", "sat"), ("the", "mat")]:
    trigram.show(ctx)
    print()

Second-order model:
<START> <START>    sum = 1.000000000000  OK
<START> the        sum = 1.000000000000  OK
cat ran            sum = 1.000000000000  OK
cat sat            sum = 1.000000000000  OK
dog ran            sum = 1.000000000000  OK
dog sat            sum = 1.000000000000  OK
on the             sum = 1.000000000000  OK
ran to             sum = 1.000000000000  OK
sat on             sum = 1.000000000000  OK
the cat            sum = 1.000000000000  OK
the dog            sum = 1.000000000000  OK
the mat            sum = 1.000000000000  OK
the park           sum = 1.000000000000  OK
the rug            sum = 1.000000000000  OK
to the             sum = 1.000000000000  OK

P(. | <START>, the):
   cat      0.5000
   dog      0.5000

P(. | on, the):
   mat      0.5000
   rug      0.5000

P(. | to, the):
   park     1.0000

P(. | the, cat):
   sat      0.6667
   ran      0.3333

P(. | cat, sat):
   on       1.0000

P(. | the, mat):
   <END>    1.0000



The extra context fixes the main weakness of the bigram model: after `<START> the` the model predicts only cat/dog, while after `on the` it predicts only mat/rug and after `to the` only park. "the cat" vs "the dog" contexts still give the same next-word distribution, since the data uses them identically.

## Part XIII: Comparing the two models

In [9]:
V_ctx = len(vocab) + 1            # tokens that can appear as context: words + <START>
V_out = len(vocab) + 1            # tokens that can be predicted: words + <END>

def compare(model, name, n=200, seed=0):
    rng = random.Random(seed)
    gens = [model.generate("sample", rng) for _ in range(n)]
    possible_contexts = V_ctx ** model.order
    nonzero_params = sum(len(d) for d in model.probs.values())
    return {
        "model": name,
        "full CPT size": possible_contexts * V_out,
        "nonzero parameters": nonzero_params,
        "observed contexts": len(model.probs),
        "unseen (zero-prob) contexts": possible_contexts - len(model.probs),
        f"distinct sentences / {n}": len(set(gens)),
        "novel distinct sentences": len(set(gens) - train_set),
        "hit max length": sum("max length" in g for g in gens),
    }, gens

r1, g1 = compare(bigram, "1st order")
r2, g2 = compare(trigram, "2nd order")
for k in r1:
    print(f"{k:<28}{str(r1[k]):>12}{str(r2[k]):>12}")

print("\nExample 1st-order sentences:")
for s in Counter(g1).most_common(8):
    print(f"   {s[1]:>3} x  {s[0]}")
print("\nExample 2nd-order sentences:")
for s in Counter(g2).most_common(8):
    print(f"   {s[1]:>3} x  {s[0]}{'' if s[0] in train_set else '   <- novel'}")

model                          1st order   2nd order
full CPT size                        121        1331
nonzero parameters                    17          19
observed contexts                     11          15
unseen (zero-prob) contexts            0         106
distinct sentences / 200              59           6
novel distinct sentences              53           0
hit max length                         8           0

Example 1st-order sentences:
    39 x  the mat
    32 x  the park
    24 x  the rug
    11 x  the cat sat on the park
     8 x  the dog sat on the rug
     7 x  the cat sat on the rug
     7 x  the dog sat on the park
     5 x  the dog sat on the mat

Example 2nd-order sentences:
    40 x  the dog sat on the mat
    38 x  the cat sat on the mat
    33 x  the dog ran to the park
    31 x  the cat ran to the park
    29 x  the dog sat on the rug
    29 x  the cat sat on the rug


In [10]:
# Scoring sentences with the chain rule under both models
tests = [
    "the cat sat on the mat",          # training sentence
    "the dog ran to the park",         # training sentence
    "the cat ran to the park",         # training sentence
    "the dog sat on the park",         # plausible, unseen combination
    "the cat sat on the dog",          # odd
    "the mat sat on the cat",          # nonsense
]
print(f"{'sentence':<28}{'P 1st order':>14}{'P 2nd order':>14}")
for s in tests:
    print(f"{s:<28}{bigram.sentence_prob(s):>14.5f}{trigram.sentence_prob(s):>14.5f}")

sentence                       P 1st order   P 2nd order
the cat sat on the mat             0.02778       0.16667
the dog ran to the park            0.01389       0.16667
the cat ran to the park            0.01389       0.16667
the dog sat on the park            0.02778       0.00000
the cat sat on the dog             0.00000       0.00000
the mat sat on the cat             0.00000       0.00000


**Comparison summary** (from the output above):

| Measure | 1st order | 2nd order |
|---|---:|---:|
| Full CPT size (contexts × outcomes) | $11 \times 11 = 121$ | $11^2 \times 11 = 1331$ |
| Nonzero parameters | 17 | 19 |
| Observed contexts | 11 | 15 |
| Unseen (zero-probability) contexts | 0 | 106 (of 121 possible pairs) |
| Distinct sentences in 200 samples | 59 | 6 |
| Novel (not in training) | 53 | 0 |
| Hit the length cap | 8 | 0 |

- **Parameters / zero contexts:** the number of nonzero parameters hardly changes (17 → 19), but the space of contexts grows from 11 to 121, and 106 of those pair-contexts are never observed, so the model has no distribution for them at all.
- **Diversity:** the bigram model is far more varied (59 distinct sentences), but much of that variety is junk: "the mat" (39×), "the park" (32×), "the rug" (24×) and run-ons that hit the length cap.
- **Coherence:** the trigram model's output is always well-formed, but it produced **only the 6 training sentences** – it has memorised the corpus. With this data every trigram context has essentially one way to continue, so it cannot generalise at all.
- **Scoring:** the bigram model gives positive probability to the unseen but reasonable "the dog sat on the park"; the trigram model gives it 0 because the triple (on, the, park) never occurred. Both give 0 to "the cat sat on the dog", since "dog → `<END>`" was never seen. The trigram model is more confident (0.167) on the training sentences – the six sentences share all of its probability mass.

**Question 12 – More context: better prediction, harder estimation.** Conditioning on more previous words removes ambiguity – $P(\cdot \mid \text{on, the})$ is much sharper and more correct than $P(\cdot \mid \text{the})$ – because the true next-word distribution really does depend on more than one word. But the CPT grows exponentially with context length: $V^{n}$ contexts for an $n$-word context, each needing its own distribution of $V-1$ free parameters. With a fixed amount of data, the counts are spread over far more rows, so most contexts are never observed (zero probability, no distribution at all) and the observed ones are estimated from very few examples (overfitting: the trigram model mostly memorises the training sentences). This is the bias–variance trade-off: less context = biased but well estimated; more context = less bias but high variance and sparsity. Smoothing, back-off, or parameter sharing (which is what neural models do) are needed to get the benefit of long context without needing exponentially much data.

## Part XIV: Connection to modern language models

Both models here and a modern LLM optimise the same objective, $\prod_t P(x_t \mid x_{<t})$ with $x_1$ conditioned on `<START>`, and both generate by sampling one token at a time from that conditional (Part IX/X). The difference is the representation: instead of a lookup table indexed by the last 1–2 words, a neural network maps the whole preceding context to a distribution over the vocabulary, via a softmax over logits. Because the network shares parameters across contexts, it gives sensible non-zero probabilities to contexts it has never seen exactly – the exact problem the trigram table could not handle.

## Part XV: Reflection on the role of the LLM

**Question 13 – Why is Approach B ("implement $P(X_t \mid X_{t-1})$ from transition counts with sampling") better than "write a Python language model"?**

- **Specifying behaviour:** Approach A lets the LLM decide what "language model" means – it might import a pretrained model, use a library, or pick greedy decoding – and then I cannot tell whether the output is right because I never said what right is. Approach B fixes the model, the estimator and the generation method, so the code can be checked against a specification.
- **Understanding the representation:** knowing that the model is a CPT keyed by context is what let me answer Questions 4–7 and spot the unseen-context `KeyError` and the tie problem.
- **Validating the implementation:** the hand-computed CPT (Question 3) gave an independent oracle; the code was tested against it rather than trusted.
- **Testing probabilistic invariants:** $\sum_v P(v \mid w) = 1$ for every context is a property any correct implementation must satisfy, and it catches whole classes of bugs (Question 8 demo) without knowing the "right answer" for each entry.
- **Implementation vs model:** greedy generation looping forever was not a bug in the probability model – the CPT was correct – but a property of the *decoding procedure* applied to that model. Keeping the two separate made the diagnosis quick.

**Question 14 – What did the Bayesian-network view add?**
1. **A representation of dependencies and independence assumptions:** the graph makes the Markov assumption explicit ($X_t$ has one parent), so the failure "after *on the*, predict *cat*" is immediately explained: the model literally cannot see "on".
2. **A factorisation of the joint:** the joint probability of a sentence is the product of the node CPTs, which gave both the generation algorithm (ancestral sampling, parents before children) and a way to score sentences (`sentence_prob`).
3. **A way to understand increasing context:** going from first to second order is adding a parent edge $X_{t-2} \to X_t$; the CPT of $X_t$ then has $V^2$ instead of $V$ rows, which explains the sparsity and zero-probability contexts measured in Part XIII.
4. **A test of the implementation against its specification:** each CPT column must be a valid distribution, and each entry is a ratio of counts. Both are checkable properties, which is how the code was validated.

### Short reflection on LLM use

The LLM generated the `NGramLM` class and the second-order modification quickly and correctly in its core – counting and normalisation passed the hand-computed CPT and the normalisation test the first time. Inspection focused on the code *around* the probabilities, and three points needed changes (marked `# FIX`): (1) a plain `self.probs[context]` lookup crashes with `KeyError` on unseen contexts; (2) `max(dist, key=dist.get)` silently breaks the cat/dog tie by dictionary order and reports a single "most probable word" where there are two; (3) a generation loop without a length cap never terminates under greedy decoding on this data. None of these would show up by running the program once on one example, and none is a problem with the CPT itself – which is why the CPT was validated separately (hand table + normalisation test).